In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [4]:
data=pd.read_csv('Churn_Modelling.csv')

In [5]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
label_encoder_gender= LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

ohe=OneHotEncoder()
geo_encoded=ohe.fit_transform(data[['Geography']]).toarray()

geo_encoded_df=pd.DataFrame(geo_encoded,columns=ohe.get_feature_names_out(['Geography']))

data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [6]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
##Define function to create model and try different hyperparameters

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='linear'))
    model.compile(optimizer='adam',loss='mse',metrics=['mae'])
    return model



In [ ]:
##Create keras regressor for grid search

model=KerasRegressor(layers=1,neurons=32,build_fn=create_model,epochs=50,batch_size=10,verbose=0)


In [11]:
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2],
    'epochs':[50,100],
}

In [15]:
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3)
grid_result=grid.fit(X_train,y_train)

ValueError: 
All the 48 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\mdtab\anaconda3\envs\tf_gpu\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\mdtab\anaconda3\envs\tf_gpu\lib\site-packages\scikeras\wrappers.py", line 1501, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
  File "c:\Users\mdtab\anaconda3\envs\tf_gpu\lib\site-packages\scikeras\wrappers.py", line 770, in fit
    self._fit(
  File "c:\Users\mdtab\anaconda3\envs\tf_gpu\lib\site-packages\scikeras\wrappers.py", line 925, in _fit
    X, y = self._initialize(X, y)
  File "c:\Users\mdtab\anaconda3\envs\tf

In [ ]:
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))